# pile-scan 1: scan a pile of pages

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daverooney/pile-scan/blob/main/notebooks/01_scan.ipynb)

Reads every image and PDF in a Google Drive folder with **Gemma 4 12B** running in llama.cpp on this notebook's GPU, and writes a transcription plus ordering clues for each page to a SQLite database on Drive. Nothing leaves the notebook: no hosted model APIs.

**It's safe to kill.** The database is copied to Drive every few pages. When Colab disconnects, reconnect and **Run all** again: it restores the database and skips every page already done.

1. **Runtime > Change runtime type**: pick a GPU (T4 on the free tier, L4 on Pro) and leave **Runtime version** on **Latest** (2026.10 or newer; older versions can't run llama.cpp's prebuilt build, and the notebook will say so).
2. Fill in the settings below.
3. **Runtime > Run all**.

llama.cpp is downloaded prebuilt (about 800 MB, under a minute). On a T4 the very first session is slower: the GPU driver compiles llama.cpp's kernels the first time each one is used, and the result is saved to Drive so later sessions skip that.

In [ ]:
#@title Settings { display-mode: "form" }
#@markdown **Folders on your Drive** (as seen after mounting)
INPUT_DIR = "/content/drive/MyDrive/pile-scan/inbox"  #@param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/pile-scan/output"  #@param {type:"string"}
DB_NAME = "pile_scan.db"  #@param {type:"string"}
RECURSIVE = True  #@param {type:"boolean"}

#@markdown **Model.** gemma-4-12b reads pages best. gemma-4-e4b is smaller and about twice as fast, and is close on plain text but weaker at understanding a page.
MODEL = "gemma-4-12b"  #@param ["gemma-4-12b", "gemma-4-e4b"]
#@markdown 12B: 4bit (7.4 GB) fits a free T4, 8bit (12.7 GB) needs an L4. E4B: both fit a T4 (5.1 / 8.2 GB).
QUANTIZATION = "4bit"  #@param ["4bit", "8bit"]
#@markdown Multi-token prediction: a small drafter guesses ahead and the model checks its guesses. Same text, usually faster. Off until it's been timed on a T4.
USE_MTP = False  #@param {type:"boolean"}
#@markdown Visual tokens per page. 1120 keeps the most detail for small print.
MAX_SOFT_TOKENS = 1120  #@param [70, 140, 280, 560, 1120] {type:"raw"}
MAX_NEW_TOKENS = 3072  #@param {type:"integer"}

#@markdown **Run control**
CHECKPOINT_EVERY = 10  #@param {type:"integer"}
#@markdown Stop after this many pages (0 = no limit). Same number gives the same pages every time.
SAMPLE_LIMIT = 0  #@param {type:"integer"}
#@markdown Read again the pages that failed or were cut off on an earlier run.
RETRY_ERRORS = False  #@param {type:"boolean"}
PDF_DPI = 200  #@param {type:"integer"}
#@markdown Release the GPU when the scan ends, so a finished run stops using compute units.
DISCONNECT_WHEN_DONE = False  #@param {type:"boolean"}

#@markdown **Caches on Drive.** The T4's compiled GPU code is always cached here. Model files only if you tick the box (costs 7-13 GB of Drive space, saves a download each session).
CACHE_DIR = "/content/drive/MyDrive/pile-scan/cache"  #@param {type:"string"}
CACHE_MODELS_ON_DRIVE = False  #@param {type:"boolean"}

#@markdown **Code to install** (the branch is useful for trying changes before they merge)
REPO = "daverooney/pile-scan"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

# Factual questions asked of every page. Keys become JSON keys in `answers_json`.
QUESTIONS = {
    "summary": "One sentence describing what this page is about.",
    "author": "Who wrote or sent this document, if shown? Otherwise null.",
    "recipient": "Who is this document addressed to, if shown? Otherwise null.",
    "organizations": "List the organizations named on the page (array of strings).",
}

In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#@title Install (uses uv and the repo's lockfile)
# If the repo is private, add a Colab secret named GITHUB_TOKEN (key icon on the left)
# holding a read-only GitHub token, and give this notebook access to it.
import shutil, subprocess, sys

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

_token = _secret("GITHUB_TOKEN")
_auth = f"{_token}@" if _token else ""
CODE_DIR = "/content/pile-scan"
shutil.rmtree(CODE_DIR, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH,
                f"https://{_auth}github.com/{REPO}.git", CODE_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
# Pinned versions from uv.lock, installed into Colab's own Python.
subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", f"{CODE_DIR}/requirements-colab.txt"], check=True)
subprocess.run(["uv", "pip", "install", "--system", "-q", "--no-deps", "--reinstall", CODE_DIR], check=True)

import pile_scan
print("pile-scan", pile_scan.__version__)

In [ ]:
#@title Check the pile (no GPU work)
from pile_scan import ScanConfig
from pile_scan.scanner import run_scan

cfg = ScanConfig(
    input_dir=INPUT_DIR, output_dir=OUTPUT_DIR, db_name=DB_NAME, recursive=RECURSIVE,
    model=MODEL, quantization=QUANTIZATION, use_mtp=USE_MTP, max_soft_tokens=int(MAX_SOFT_TOKENS),
    cache_dir=CACHE_DIR, cache_models_on_drive=CACHE_MODELS_ON_DRIVE,
    max_new_tokens=MAX_NEW_TOKENS, checkpoint_every=CHECKPOINT_EVERY,
    sample_limit=SAMPLE_LIMIT or None, retry_errors=RETRY_ERRORS, pdf_dpi=PDF_DPI,
    questions=QUESTIONS,
)
cfg.validate()
# Hashes every file and counts pages. Hashes are cached, so later runs only
# hash new or changed files.
inventory = run_scan(cfg, None)
print(f"{inventory.pages_total_known} pages in {inventory.files_seen} files; "
      f"{inventory.pages_skipped} already done.")

In [ ]:
#@title Check the runtime and get llama-server (prebuilt, no compiling)
from pile_scan import llama

# Stops here with a fix if this runtime is too old or has no GPU.
gpu = llama.check_environment(cfg)
server_bin = llama.ensure_server_binary(cfg)

# GPU code the driver compiled in earlier sessions (T4 only; an L4 doesn't need it).
jit = llama.JitCache(cfg, gpu)
if gpu.needs_jit:
    jit.restore()
print("llama-server sees", llama.verify_server(server_bin, env=jit.env()))

In [ ]:
#@title Download the model
model_files = llama.ensure_model_files(cfg, token=_secret("HF_TOKEN"))
print(model_files)

In [ ]:
#@title Start the server and scan
# Safe to stop at any time (Runtime > Interrupt execution): progress is saved.
from pathlib import Path
from pile_scan.scanner import run_scan

if gpu.needs_jit and not jit.drive_file.exists():
    print("First session on this GPU: starting the server and the first pages will be slow "
          "while the driver compiles GPU code. It's saved to Drive as it goes.")
server = llama.LlamaServer(llama.server_args(server_bin, model_files, cfg),
                           port=cfg.server_port, log_path=Path(cfg.local_dir) / "llama-server.log",
                           env=jit.env())
try:
    server.start()
    jit.save()
    reader = llama.LlamaPageReader(server, cfg, after_read=jit.save)
    summary = run_scan(cfg, reader, gpu_name=gpu.name)
finally:
    server.stop()
    jit.save()

if DISCONNECT_WHEN_DONE and summary.end_reason != "interrupted":
    drive.flush_and_unmount()   # make sure the last checkpoint reaches Drive
    from google.colab import runtime
    runtime.unassign()

## Look at the results

The database on Drive is a normal SQLite file. Main tables:

- `pages`: one row per page with `ocr_text`, the ordering clues (`doc_title`, `page_number_printed`, `primary_date`, `first_line`, `last_line`, `running_header`, `ends_mid_sentence`, ...) and `answers_json`.
- `files`: one row per distinct file, keyed by its SHA-256; `file_paths` lists every path it was found at.
- `runs`: one row per scan run with its settings.

In [ ]:
#@title Recent pages
import sqlite3, pandas as pd
con = sqlite3.connect(str(cfg.local_db_path))
pd.read_sql("""
    SELECT f.source_path, p.page_index + 1 AS page, p.status, p.doc_title, p.doc_type,
           p.page_number_printed, p.primary_date, p.truncated, p.loop_detected, p.retry_mode,
           ROUND(p.elapsed_s, 1) AS secs,
           substr(p.ocr_text, 1, 120) AS text_start
    FROM pages p JOIN files f USING (file_sha256)
    ORDER BY p.created_at DESC LIMIT 25
""", con)

## Optional: compare two models or settings

For example, the 12B against the E4B on a T4:

1. Set `SAMPLE_LIMIT` to 20, `MODEL` to `gemma-4-12b`, `DB_NAME` to `sample_12b.db`, and Run all.
2. Set `MODEL` to `gemma-4-e4b`, `QUANTIZATION` to `8bit`, `DB_NAME` to `sample_e4b.db`, and Run all.
3. Point the cell below at the two databases and run it. Both runs read the same 20 pages, because files are visited in sorted order.

The same works for 4-bit against 8-bit, or `USE_MTP` on against off (MTP should give identical text, only faster).

Numbers (dates, amounts, IDs) are where lower precision tends to slip first, so look hardest at `numbers_match`.

In [ ]:
#@title Compare two sample runs
DB_A = "/content/drive/MyDrive/pile-scan/output/sample_12b.db"  #@param {type:"string"}
DB_B = "/content/drive/MyDrive/pile-scan/output/sample_e4b.db"  #@param {type:"string"}
import pandas as pd
from pile_scan import compare
rows = compare.compare(DB_A, DB_B)
print(compare.summarize(rows))
pd.DataFrame(rows).sort_values("text_similarity")